In [1]:
!pip install transformers datasets peft accelerate bitsandbytes --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 19.3 MB/s eta 0:00:00


In [ ]:
!pip install Groq --quite

In [ ]:
import json
from transformer import AutoModelForCausalLM, AutoTokenizer, pipeline
from perf import LoraConfig, get_perf_model, TaskType, prepare_model_for_kbit_training
from datasets import load_dataset, Dataset, DatasetDisct
from sklearn.model_selection import train_test_split

In [ ]:
from groq import Groq
client = Groq(api_key= )

In [ ]:
source_text = """
1. کتاب «آموزش برنامه‌نویسی C#» (سیدمجتبی موسوی)
2. وب‌سایت‌های آموزشی barnamenevis.org و tutsplus فارسی
3. مستندات Microsoft Learn در بخش C#
"""

In [ ]:
def generate_questions(source_text, num_questions=20):
    prompt = f"""
    شما یک مدرس هستید. متن زیر را بخوانید و {num_questions} سوال چهارگزینه‌ای با جواب درست ایجاد کنید.
    هر سوال به فارسی باشد و گزینه‌ها کاملاً متفاوت. فرمت JSON خروجی به شکل زیر باشد:
    [
      {{
        "question": "متن سوال",
        "options": ["گزینه 1", "گزینه 2", "گزینه 3", "گزینه 4"],
        "answer": "گزینه صحیح"
      }}
    ]
    متن منبع:
    {source_text}
    """
    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )
    text = response.choices[0].message.content
    return json.loads(text)

In [ ]:
questions = generate_questions(source_text, num_questions=10)

In [ ]:
with open("all_questions.json", "w", encoding="utf-8") as f:
    json.dump(questions, f, ensure_ascii=False, indent=2)


In [ ]:
train_data, test_data = train_test_split(questions, test_size=0.2, random_state=42)

with open("train.json", "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)
with open("test.json", "w", encoding="utf-8") as f:
    json.dump(test_data, f, ensure_ascii=False, indent=2)


In [ ]:
model_name = "HooshvareLab/bert-fa-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none"
)
model = get_peft_model(model, lora_config)

In [ ]:
def preprocess(example):
    prompt = f"سوال: {example['question']}\nگزینه‌ها: {example['options']}\nپاسخ:"
    label = example['answer']
    enc = tokenizer(prompt, truncation=True, padding="max_length", max_length=128)
    enc["labels"] = tokenizer(label, truncation=True, padding="max_length", max_length=32)["input_ids"]
    return enc

train_dataset = Dataset.from_list(train_data).map(preprocess)
test_dataset = Dataset.from_list(test_data).map(preprocess)

dataset = DatasetDict({"train": train_dataset, "test": test_dataset})

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./lora_finetuned",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    learning_rate=3e-4,
    fp16=True,
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"]
)

trainer.train()

In [ ]:
def evaluate_model(model, dataset):
    correct = 0
    for example in dataset:
        prompt = f"سوال: {example['question']}\nگزینه‌ها: {example['options']}\nپاسخ:"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=32)
        pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if example['answer'] in pred:
            correct += 1
    return correct / len(dataset)

accuracy = evaluate_model(model, test_data)
print(f"accuracy: {accuracy*100:.2f}%")